In [0]:
# ================================================================
# NOTEBOOK: nb_gold_customer_cohorts
# PURPOSE:  Monthly customer cohort retention analysis
# RUN:      Monthly (1st of every month)
# READS:    silver/orders + silver/customers
# WRITES:   gold/customer_cohorts/
# GRAIN:    1 row per (CohortMonth x OrderMonth)
#
# WHAT IS COHORT ANALYSIS?
#   Customers are grouped by the month of their first delivered order.
#   Each group is called a cohort.
#
#   We then calculate how many customers from each cohort purchased
#   again in the following months.
#
# EXAMPLE:
#   January cohort size = 100 customers
#   Month 0 = 100 active customers = 100% retention
#   Month 1 = 60 active customers  = 60% retention
#   Month 2 = 45 active customers  = 45% retention
#
# POWER BI:
#   Rows    = Cohort month
#   Columns = MonthIndex (0, 1, 2, 3...)
#   Values  = RetentionRate
# ================================================================

# ================================================================
# IMPORTS
# ================================================================


from pyspark.sql import functions as F
from pyspark.sql.functions import (col, countDistinct,lit, when)


# ================================================================
# CONFIGURATION
# ================================================================


STORAGE = ("abfss://source@stshopsensedevhj.dfs.core.windows.net")
ORDERS_PATH = (f"{STORAGE}/silver/orders/")

CUSTOMERS_PATH = (f"{STORAGE}/silver/customers/")



GOLD_PATH = (
    f"{STORAGE}/gold/customer_cohorts/"
)

# ================================================================
# STEP 1: READ SILVER ORDERS
# ================================================================
#
# Only delivered orders are considered real purchases.
#
# CDC-deleted orders are excluded using _is_deleted.
# If _is_deleted does not exist, no delete filter is applied.
# ================================================================



orders = (
    spark.read
    .format("delta")
    .load(ORDERS_PATH)
)


# ------------------------------------------------
# Validate required order columns
# ------------------------------------------------

required_order_column = [ "OrderID", "CustomerID", "OrderDate", "NetAmount", "IsDelivered"]


missing_order_columns = [
    column_name
    for column_name in required_order_column
    if column_name not in orders.columns
]

if missing_order_columns:
    raise ValueError(
        f"Missing required order columns: "
        f"{missing_order_columns}"
        )


# ------------------------------------------------
# Filter delivered and active orders
# ------------------------------------------------

orders = (
    orders
    .filter(col("IsDelivered") == True)
)


if "_is_deleted" in orders.columns:
    orders = orders.filter(F.coalesce(col("_is_deleted"),lit(False)) == False)




# ------------------------------------------------
# Prepare only required columns
# ------------------------------------------------

orders = (
    orders
    .select(
        col("OrderID"),
        col("CustomerID"),
        col("OrderDate").cast("date").alias("OrderDate"),
        col("NetAmount").cast("double").alias("NetAmount")
    )
)


# ================================================================
# STEP 2: READ CURRENT CUSTOMER RECORDS
# ================================================================
#
# Customer profile columns represent the customer's CURRENT profile.
#
# Example:
# If a customer joined as Regular but later became VIP,
# their cohort profile will currently show them as VIP.
# ================================================================


customers = (
    spark.read.format("delta")
    .load(CUSTOMERS_PATH)
)



# ------------------------------------------------
# Validate required customer columns
# ------------------------------------------------


required_customer_columns = ["CustomerID", "Segment", "IsPrimeBool"]


missing_customer_columns = [
column_name
for column_name in required_customer_columns
if column_name not in customers.columns
]

if missing_customer_columns:
    raise ValueError(
        f"Missing required customer columns: "
        f"{missing_customer_columns}"
        )




# ------------------------------------------------
# Keep current SCD Type 2 customer records
# ------------------------------------------------


if "is_current" in customers.columns:
    customers = (
        customers
        .filter(col("is_current") == True)
    )

    
customers = (
    customers
    .select(
        col("CustomerID"),
        col("Segment"),
        col("IsPrimeBool"))
        .dropDuplicates(["CustomerID"])
)



# ================================================================
# STEP 3: SOURCE COUNTS
# ================================================================


delivered_order_count = (
    orders
    .count()
)


current_customer_count = (
    customers
    .count()
)


print(
    f"[READ] Delivered active orders: "
    f"{delivered_order_count}"
)


print(
    f"[READ] Current customers: "
    f"{current_customer_count}"
)



if delivered_order_count  == 0:
    raise ValueError(
        "No delivered active orders were found."
        )



# ================================================================
# STEP 4: FIND EACH CUSTOMER'S FIRST ORDER
# ================================================================
#
# FirstOrderDate:
#   Earliest delivered order date for the customer.
#
# CohortMonth:
#   First day of the month in which the customer first ordered.
#
# Example:
#   FirstOrderDate = 17-Jan-2024
#   CohortMonth    = 01-Jan-2024
# ================================================================


first_order = (
    orders
    .groupBy("CustomerID")
    .agg(
        F.min("OrderDate").alias("FirstOrderDate")
        )
        .withColumn("CohortMonth", F.date_trunc("month",
        col("FirstOrderDate")
        ).cast("date")
        )
        .withColumn("CohortYear",F.year(col("CohortMonth"))
        )
        .withColumn("CohortMonthNum",F.month(col("CohortMonth"))
        )
        .withColumn("CohortLabel",F.date_format(col("CohortMonth"),"MMM-yyy"))

)


unique_customer_count  = (
    first_order
    .count()
)

print(
    f"\n[COHORT] Customers assigned to cohorts: "
    f"{unique_customer_count}"
)


print("\n[COHORTS IDENTIFIED]")

(
first_order
.groupBy(
    "CohortMonth",
    "CohortLabel"
).agg(
    countDistinct("CustomerID").alias("CohortCustomers")
).orderBy("CohortMonth")
.show(truncate=False)

)

# ================================================================
# STEP 5: TAG EVERY ORDER WITH CUSTOMER'S COHORT
# ================================================================
#
# OrderMonth:
#   Month in which the order was placed.
#
# MonthIndex:
#   Number of months since the customer's first-order month.
#
# Examples:
#   CohortMonth = January, OrderMonth = January  → MonthIndex 0
#   CohortMonth = January, OrderMonth = February → MonthIndex 1
#   CohortMonth = January, OrderMonth = March    → MonthIndex 2
# ================================================================



orders_with_cohort = (
    orders
    .join(
        first_order.select(
            "CustomerID",
            "CohortMonth",
            "CohortYear",
            "CohortMonthNum",
            "CohortLabel"
        ),
        on="CustomerID",
        how="inner"
    )
    .withColumn(
        "OrderMonth",
        F.date_trunc(
            "month",
            F.col("OrderDate").cast("date")
        )
    )
    .withColumn(
        "MonthIndex",
        (
            (
                F.year(F.col("OrderMonth")) -
                F.year(F.col("CohortMonth"))
            ) * 12
            +
            (
                F.month(F.col("OrderMonth")) -
                F.month(F.col("CohortMonth"))
            )
        ).cast("integer")
    )
    .filter(F.col("MonthIndex") >= 0)
)




# ================================================================
# STEP 6: CALCULATE COHORT ACTIVITY
# ================================================================
#
# ActiveCustomers:
#   Distinct customers from the cohort who ordered in that month.
#
# TotalOrders:
#   Distinct delivered orders placed by those customers.
#
# TotalRevenue:
#   Revenue generated by the cohort during that month.
#
# AvgOrderValue:
#   Total revenue divided by distinct orders.
# ================================================================


cohort_activity = (
    orders_with_cohort
    .groupBy(
        "CohortMonth",
        "CohortLabel",
        "CohortYear",
        "CohortMonthNum",
        "OrderMonth",
        "MonthIndex")
        .agg(
            F.countDistinct("CustomerID").alias("ActiveCustomers"),
            F.countDistinct("OrderID").alias("TotalOrders"),
            F.round(F.sum(F.coalesce(F.col("NetAmount"), F.lit(0.0))), 2).alias("TotalRevenue")
        )
            .withColumn("AvgOrderValue",
            when(
                col("TotalOrders") > 0,
                F.round(
                    col("TotalRevenue") /
                    col("TotalOrders"),
                    2
                )
            ).otherwise(F.lit(0.0))
        )
    )



# ================================================================
# STEP 7: CALCULATE ORIGINAL COHORT SIZE
# ================================================================
#
# CohortSize:
#   Total customers whose first delivered order occurred in that month.
#
# Calculating cohort size directly from first_order is safer than
# deriving it from MonthIndex 0 activity.
# ================================================================

cohort_sizes = (
    first_order
    .groupBy(
        "CohortMonth")
        .agg(
            countDistinct("CustomerID").alias("CohortSize")
        )
    )



# ================================================================
# STEP 8: CALCULATE RETENTION AND INACTIVITY
# ================================================================
#
# RetentionRate:
#   ActiveCustomers / CohortSize × 100
#
# InactiveRate:
#   100 - RetentionRate
#
# InactiveRate is more accurate than calling it permanent churn,
# because a customer may skip one month and return later.
# ================================================================


cohort_retention = (
    cohort_activity
    .join(
        cohort_sizes,
        on="CohortMonth",
        how="left"
    )
    .withColumn(
        "RetentionRate",
        when(
            col("CohortSize") > 0,
            F.round(
                (col("ActiveCustomers") / col("CohortSize")) * 100,
                2
            )
        ).otherwise(lit(0.0))
    )
    .withColumn(
        "InactiveRate",
        F.round(
            lit(100.0) - col("RetentionRate"),
            2
        )
    )
)
    

# ================================================================
# STEP 9: CREATE COHORT CUSTOMER PROFILE
# ================================================================
#
# These columns describe the CURRENT profile of customers
# belonging to each cohort.
# ================================================================


cohort_profile = (
    first_order
    .join(customers, on="CustomerID", how="inner")
    .groupBy(
        "CohortMonth",
        "CohortLabel"
    )
    .agg(
        F.sum(
            F.when(
                F.trim(F.col("Segment")) == "REGULAR",
                1
            ).otherwise(0)
        ).alias("RegularCount"),

        F.sum(
            F.when(
                F.upper(
                    F.trim(
                        col("Segment")
                    )
                ) == "PREMIUM",
                1
            ).otherwise(0)
        ).alias("PremiumCount"),

        F.sum(
            F.when(
                F.trim(F.col("Segment")) == "VIP",
                1
            ).otherwise(0)
        ).alias("VIPCount"),

        F.sum(
            F.when(
                F.col("IsPrimeBool") == True,
                1
            ).otherwise(0)
        ).alias("PrimeCount"),

        F.countDistinct("CustomerID").alias("TotalInCohort")
    )
    .withColumn(
        "PrimePct",
        F.when(
            F.col("TotalInCohort") > 0,
            F.round(
                F.col("PrimeCount") /
                F.col("TotalInCohort") * 100,
                1
            )
        ).otherwise(F.lit(0.0))
    )
)



# ================================================================
# STEP 10: CREATE FINAL GOLD DATAFRAME
# ================================================================


gold_df = (
    cohort_retention
    .join(
        cohort_profile.select(
            "CohortMonth",
            "RegularCount",
            "PremiumCount",
            "VIPCount",
            "PrimeCount",
            "PrimePct"
        ),
        on = "CohortMonth",
        how = "left"
    )
    .withColumn(
        "_gold_load_ts",
        F.current_timestamp()
    ).select(
        "CohortMonth",
        "CohortLabel",
        "CohortYear",
        "CohortMonthNum",
        "OrderMonth",
        "MonthIndex",

        "CohortSize",
        "ActiveCustomers",
        "RetentionRate",
        "InactiveRate",
        "TotalOrders",
        "TotalRevenue",
        "AvgOrderValue",

        "PrimeCount",
        "PrimePct",
        "RegularCount",
        "PremiumCount",
        "VIPCount",

        "_gold_load_ts"
    )

)


# ================================================================
# STEP 11: DATA QUALITY VALIDATIONS
# ================================================================   


gold_row_count = (
    gold_df
    .count()
)

# Negative MonthIndex check
negative_month_indexes = (
    gold_df
    .filter(
        col("MonthIndex") < 0
    )
    .count()
)

# Invalid retention rate check
invalid_retention_rows = (
    gold_df
    .filter(
        (col("RetentionRate") < 0) |
        (col("RetentionRate") > 100)
    )
    .count()
)
    
# Month 0 should be 100%

month_zero_not_100 = (
    gold_df
    .filter(
        (col("MonthIndex") == 0) &
        (col("RetentionRate") != 100)
    )
    .count()
)

duplicate_grain_rows = (
    gold_df
    .groupBy("CohortMonth", "OrderMonth")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("\n[DATA QUALITY VALIDATION]")

print(f"Gold rows: {gold_row_count}")


print(f"Negative MonthIndex rows: "
      f"{negative_month_indexes}"
)



print(
    f"Invalid RetentionRate rows: "
    f"{invalid_retention_rows}"
)


print(
    f"Month 0 rows not equal to 100%: "
    f"{month_zero_not_100}"
)


print(
    f"Duplicate grain combinations: "
    f"{duplicate_grain_rows}"
)

if negative_month_indexes > 0:
    raise ValueError(
        "Negative MonthIndex values found."
    )



if invalid_retention_rows > 0:
    raise ValueError(
        "RetentionRate outside the range 0 to 100."
        )
    

if duplicate_grain_rows > 0:
    raise ValueError(
        "Duplicate CohortMonth × OrderMonth rows found."
        )




# ================================================================
# STEP 12: WRITE GOLD DELTA TABLE
# ================================================================


(
    gold_df
    .write
    .format("delta")
    .option("overwriteSchema", "true")
    .partitionBy(
    "CohortYear",
    "CohortMonthNum")
    .mode('overwrite').save(GOLD_PATH)
)


print(
    f"\n[DONE] gold/customer_cohorts/ written: "
    f"{gold_row_count} rows"
)


# ================================================================
# STEP 13: COHORT RETENTION HEATMAP PREVIEW
# ================================================================


print("\n[COHORT RETENTION HEATMAP]")


print("Each row = customers who first purchased in that month")


print("Each column = months after their first purchase\n")


cohort_pivot = (
    gold_df

    # Show only MonthIndex 0 to 5
    .filter(
        col("MonthIndex").between(0, 5)
    )

    # Create one row per cohort
    .groupBy(
        "CohortMonth",
        "CohortLabel",
        "CohortSize"
    )

    # Convert MonthIndex values into columns
    .pivot(
        "MonthIndex",
        [0, 1, 2, 3, 4, 5]
    )

    # Put RetentionRate inside every MonthIndex column
    .agg(
        F.first("RetentionRate")
    )

    # Arrange cohorts from oldest to newest
    .orderBy(
        "CohortMonth"
    )

    # Show only useful columns
    .select(
        "CohortLabel",
        "CohortSize",
        "0",
        "1",
        "2",
        "3",
        "4",
        "5"
    )
)


cohort_pivot.show(
    20,
    truncate=False
)
        


# ================================================================
# STEP 14: AVERAGE RETENTION BY MONTH INDEX
# ================================================================
#
# AvgCohortRetention:
#   Every cohort receives equal importance.
#
# WeightedRetentionRate:
#   Larger cohorts receive more importance.
# ================================================================


print("\n[AVERAGE RETENTION BY MONTH INDEX]")

average_retention = (
    gold_df
    .groupBy("MonthIndex")
    .agg(
        # Every cohort has equal importance
        F.round(
            F.avg("RetentionRate"),
            1
        ).alias("AvgCohortRetention"),

        # Larger cohorts have more importance
        F.round(
            F.sum("ActiveCustomers") /
            F.sum("CohortSize") * 100,
            1
        ).alias("WeightedRetentionRate"),

        # Average revenue produced by a cohort
        F.round(
            F.avg("TotalRevenue"),
            0
        ).alias("AvgRevenuePerCohort"),

        # Simple average of cohort-level AOV values
        F.round(
            F.avg("AvgOrderValue"),
            0
        ).alias("AvgOrderValue"),

        # Active customers across all cohorts
        F.sum("ActiveCustomers")
        .alias("TotalActiveCustomers")
    )
    .orderBy("MonthIndex")
    .limit(20)
)

average_retention.show(truncate=False)



# ================================================================
# STEP 15: COHORT SIZE SUMMARY
# ================================================================  

print("\n[COHORT SIZE SUMMARY]")

(
    gold_df
    .filter(
        col("MonthIndex") == 0
    )
    .select(
        "CohortMonth",
        "CohortLabel",
        "CohortSize",
        "PrimeCount",
        "PrimePct",
        "RegularCount",
        "PremiumCount",
        "VIPCount")
        .orderBy("CohortMonth")
        .drop("CohortMonth")
        .show(truncate = False)
)




# ================================================================
# STEP 16: BEST MONTH-1 COHORTS
# ================================================================

print("\n[BEST COHORTS — HIGHEST MONTH-1 RETENTION]")

(
gold_df
.filter(
    col("MonthIndex") == 1
).select(
    "CohortMonth",
    "CohortLabel",
    "CohortSize",
    "ActiveCustomers",
    "RetentionRate"
).orderBy(
        col("RetentionRate").desc(),
        col("CohortMonth").asc()
)
.drop("CohortMonth")
.limit(10)
.show(truncate=False)
)



# ================================================================
# STEP 17: WORST MONTH-1 COHORTS
# ================================================================

print("\n[WORST COHORTS — LOWEST MONTH-1 RETENTION]")

(
    gold_df
    .filter(
        col("MonthIndex") == 1
    ).select(
        "CohortMonth",
        "CohortLabel",
        "CohortSize",
        "ActiveCustomers",
        "RetentionRate"
    )
    .orderBy(
        col("RetentionRate").asc(),
        col("CohortMonth").asc()
    ).drop("CohortMonth")
    .limit(10)
    .show(truncate=False)
)


# ================================================================
# STEP 18: DISPLAY FINAL GOLD OUTPUT
# ================================================================


display(
    gold_df
    .select(
        "CohortMonth",
        "CohortLabel",
        "CohortSize",
        "MonthIndex",
        "OrderMonth",
        "ActiveCustomers",
        "RetentionRate",
        "InactiveRate",
        "TotalOrders",
        "TotalRevenue",
        "AvgOrderValue",
        "PrimePct",
        "RegularCount",
        "PremiumCount",
        "VIPCount"
).orderBy(
    "CohortMonth",
    "MonthIndex")
    .limit(30)
)






[READ] Delivered active orders: 1453
[READ] Current customers: 500

[COHORT] Customers assigned to cohorts: 477

[COHORTS IDENTIFIED]
+-----------+-----------+---------------+
|CohortMonth|CohortLabel|CohortCustomers|
+-----------+-----------+---------------+
|2024-01-01 |Jan-2024   |194            |
|2024-02-01 |Feb-2024   |109            |
|2024-03-01 |Mar-2024   |86             |
|2024-04-01 |Apr-2024   |42             |
|2024-05-01 |May-2024   |31             |
|2024-06-01 |Jun-2024   |15             |
+-----------+-----------+---------------+


[DATA QUALITY VALIDATION]
Gold rows: 21
Negative MonthIndex rows: 0
Invalid RetentionRate rows: 0
Month 0 rows not equal to 100%: 0
Duplicate grain combinations: 0

[DONE] gold/customer_cohorts/ written: 21 rows

[COHORT RETENTION HEATMAP]
Each row = customers who first purchased in that month
Each column = months after their first purchase

+-----------+----------+-----+-----+-----+-----+-----+-----+
|CohortLabel|CohortSize|0    |1    |2  

CohortMonth,CohortLabel,CohortSize,MonthIndex,OrderMonth,ActiveCustomers,RetentionRate,InactiveRate,TotalOrders,TotalRevenue,AvgOrderValue,PrimePct,RegularCount,PremiumCount,VIPCount
2024-01-01,Jan-2024,194,0,2024-01-01T00:00:00.000Z,194,100.0,0.0,260,6725527.25,25867.41,59.3,109,38,47
2024-01-01,Jan-2024,194,1,2024-02-01T00:00:00.000Z,73,37.63,62.37,93,2325305.93,25003.29,59.3,109,38,47
2024-01-01,Jan-2024,194,2,2024-03-01T00:00:00.000Z,79,40.72,59.28,105,2845353.01,27098.6,59.3,109,38,47
2024-01-01,Jan-2024,194,3,2024-04-01T00:00:00.000Z,64,32.99,67.01,73,1801174.89,24673.63,59.3,109,38,47
2024-01-01,Jan-2024,194,4,2024-05-01T00:00:00.000Z,76,39.18,60.82,98,2409472.68,24586.46,59.3,109,38,47
2024-01-01,Jan-2024,194,5,2024-06-01T00:00:00.000Z,62,31.96,68.04,77,1828555.18,23747.47,59.3,109,38,47
2024-02-01,Feb-2024,109,0,2024-02-01T00:00:00.000Z,109,100.0,0.0,140,3638586.14,25989.9,56.9,65,25,19
2024-02-01,Feb-2024,109,1,2024-03-01T00:00:00.000Z,37,33.94,66.06,52,1423412.24,27373.31,56.9,65,25,19
2024-02-01,Feb-2024,109,2,2024-04-01T00:00:00.000Z,38,34.86,65.14,50,1146653.25,22933.07,56.9,65,25,19
2024-02-01,Feb-2024,109,3,2024-05-01T00:00:00.000Z,37,33.94,66.06,47,1273305.74,27091.61,56.9,65,25,19
